# DT401 Week 7 Section 2: Analysing Data Through Time

**Module:** Data Fundamentals  
**Week:** 7 of 7  
**Section:** 2 of 2  
**Estimated Time:** 50 minutes  

---

## Learning Objectives

By the end of this notebook, you will be able to:
- Identify and explain time series components (trend, seasonal, cyclical, noise)
- Decompose time series data to understand underlying patterns
- Apply Holt-Winters exponential smoothing to build forecasts
- Understand when to use ARIMA models for more complex temporal patterns
- Design time series forecasting approaches for workplace scenarios

---

## Context Recap

In Section 1, you built regression models using weather variables measured at a single point in time to forecast event attendance. Time series analysis extends this by examining how data evolves across time - identifying trends, seasonal cycles, and temporal patterns that enable forecasting even without external predictors.

Understanding time is crucial for business planning: weekly cycles, seasonal peaks, long-term growth trends, and cyclical economic patterns all affect decisions about resources, staffing, and strategy.

**Dataset:** Open-Meteo weather observations, London, UK (51.49°N, 0.16°W), January 2023–January 2026.

This notebook builds directly on the regression work in Section 1. There, you predicted attendance from weather conditions measured at a single moment. Here, you will examine how data changes *over time*, identifying the patterns baked into a time series and using them to forecast future values. 

The colour-coded boxes you saw in Section 1 are used in the same way throughout this notebook: yellow for reminders, blue for key insights, and green for tasks.


---

## Part 1: Time Series Components

In Section 1, each row of data was an independent observation - one hour of weather measurements, with no assumed connection to the hours before or after. Time series data works differently. Here, each observation's value is partly shaped by the values that came before it. This **temporal ordering** is what makes specialised time series techniques possible, and useful.

To see why this matters, consider a charity monitoring monthly donation income. January income is almost always lower than December income - not because of any external event, but because the same pattern repeats every year. A model that ignores this structure will consistently make poor forecasts. One that captures it can plan months ahead.


### The Four Components

Any time series can be decomposed into four kinds of elements. Understanding which components are present in your data is the essential first step before choosing a forecasting method.

| Component | Definition | Business Example |
|:----------|:-----------|:----------------|
| **Trend** | Long-term increase or decrease | NHS appointments growing year-on-year due to ageing population |
| **Seasonal** | Repeating patterns with a fixed period | Retail sales spiking every December; library usage lower in summer |
| **Cyclical** | Repeating patterns without a clear fixed period | Economic cycles driving charity donation levels |
| **Noise** | Random, unpredictable variation | One-off events; measurement error |

The diagram below shows how these four components combine to produce the observed data and how decomposition pulls them apart again.


```mermaid
flowchart TD
    A[Observed Data] --> B[Decomposition]
    B --> C[Trend: Long-term direction]
    B --> D[Seasonal: Repeating cycles]
    B --> E[Residual: Unpredictable noise]
    C --> F["Reconstruct: Trend + Seasonal + Residual ≈ Observed"]
    D --> F
    E --> F
```


Decomposition separates the observed data into its predictable components (trend and seasonal) and its unpredictable variation (residual). Forecasting models exploit the predictable components. The residual sets the floor on how accurate any forecast can ever be. No model can predict genuinely random variation.


### Workplace Applications

Time series analysis is used across every sector. The table below gives a sense of the range of uses. As you work through this notebook, consider where your own organisation sits.

| Sector | Typical Time Series | Forecast Use |
|:-------|:-------------------|:------------|
| Healthcare | Daily A&E admissions, weekly GP appointments | Staffing, capacity planning |
| Retail | Daily transactions, weekly footfall | Stock ordering, staff scheduling |
| Finance | Monthly revenue, daily transaction volumes | Budget planning, fraud detection |
| Local Government | Monthly planning applications, seasonal service requests | Resource allocation |
| Charity | Monthly donation income, volunteer sign-ups | Campaign timing, coordinator staffing |
| Manufacturing | Daily production output, quality defect rates | Maintenance scheduling, capacity planning |


<div class="alert alert-block alert-warning">
<b>Reminder:</b><br>
Time series forecasting is one of the most common forms of practical data analysis. NHS trusts forecast winter pressures, retailers forecast Black Friday demand, and councils forecast peak-season park usage. These all use the same fundamental techniques you will practise here.
</div>

### Importing Libraries and Setting Up the Environment

Let's start by checking we can import the packages and functions we need for this notebook. 

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import FloatSlider, IntSlider, Layout, VBox, interactive_output
from IPython.display import display

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

sns.set_theme(style='whitegrid')

print("Libraries imported successfully. Ready to load and prepare the data.")

### Loading the Data

The next cell loads the same London weather dataset from Section 1. This time, however, we do two extra things: 

1. We convert the date column to a proper **datetime format**. This allows Python to recognise these as dates, not just text strings, which is essential for manipulating the data in time-aware ways.
2. We set it as the **index** of the DataFrame: the row labels. Setting dates as the index is what tells pandas that our data is ordered by time, which unlocks the specialised time series functions we will use throughout this notebook.

The cleaning steps are the same as Section 1: 

1. Skip the header rows
2. Clean column names
3. Convert text columns to numbers. 

Run the cell and check the output. You should see roughly 26,000 hourly observations covering January 2023 to January 2026.

In [ ]:
# Load and prepare weather data
weather_data = pd.read_csv('../open-meteo-51.49N0.16W23m.csv', skiprows=3, dtype=str)  # Skip the first 3 metadata rows; read all columns as strings to avoid mixed-type warnings before numeric conversion
weather_data.columns = weather_data.columns.str.split(' ').str[0].str.strip()  # Clean column names: remove units like '(°C)' and extra spaces
weather_data = weather_data.rename(columns={'time': 'date'})
weather_data = weather_data[weather_data['date'].str.match(r'^\d{4}-\d{2}-\d{2}T', na=False)].reset_index(drop=True)  # Keep only rows with valid date formats

numeric_cols = weather_data.columns[1:]
weather_data[numeric_cols] = weather_data[numeric_cols].apply(pd.to_numeric, errors='coerce')  # Convert text to numbers; unconvertible values become NaN (missing)

weather_data['date'] = pd.to_datetime(weather_data['date'])  # Convert the date column to a proper datetime type
weather_data = weather_data.set_index('date')  # Set dates as the row index so pandas treats this as time-ordered data

print(f"Dataset: {len(weather_data):,} hourly observations")
print(f"Date range: {weather_data.index.min().date()} to {weather_data.index.max().date()}")

### Visualising the Temperature Series

The first step in any time series analysis is to visualise the data. This helps us identify the components we will need to model later on. 

The next cell plots the hourly temperature data across the full three-year period. You should see a clear seasonal pattern, with peaks in summer and troughs in winter. If you look closely, you may also see weekly and daily cycles, the temperature tends to be lower at night, and slightly higher on weekends than weekdays.

In [ ]:
ax = weather_data['temperature_2m'].plot(
    figsize=(14, 4),
    alpha=0.7,
    color='steelblue',
    linewidth=0.8,
    ylabel='Temperature (°C)',
    xlabel='Date',
    title='Daily Average Temperature: London 2023–2026',
    grid=True
    )

ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

For our purposes, the most important pattern is the annual seasonal cycle. To see this more clearly, we need to reduce the noise in the data. As this data is from the UK, it also allows us to forget about the effect of daylight saving time, when a day is 23 hours long in spring and 25 hours long in autumn. This would otherwise create complications in our analysis, as the number of observations per day would not be consistent across the year.

The next cell aggregates (combines) hourly temperature readings into daily averages using `.resample('D').mean()`. You can think of this as calculating a daily summary: instead of 24 separate temperature readings, we get one average temperature per day. 

With roughly 1,000 daily values across three years, the annual seasonal cycle becomes clearly visible.


In [ ]:
# Aggregate to daily average temperature
daily_temp = weather_data['temperature_2m'].resample('D').mean()

# Visualise full time series
ax = daily_temp.plot(
    figsize=(14, 4),
    alpha=0.7,
    color='steelblue',
    linewidth=0.8,
    ylabel='Temperature (°C)',
    xlabel='Date',
    title='Daily Average Temperature: London 2023–2026',
    grid=True
)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

print(f"Daily observations: {len(daily_temp)}")
print(f"Temperature range: {daily_temp.min():.1f}°C to {daily_temp.max():.1f}°C")


<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We aggregated 26,000+ hourly observations to daily averages using `.resample('D').mean()`
- The chart clearly shows **seasonal cycles**: summer peaks and winter troughs repeating annually
- You can also see **noise**: the day-to-day variation within each season

**Key lesson:** Time series analysis requires regular spacing. We transformed irregular hourly readings to consistent daily averages to enable seasonal analysis.
</div>

### Time Series Decomposition

The chart above shows trend, seasonal, and noise all mixed together. [Decomposition](https://otexts.com/fpp3/decomposition.html) is the process of separating the time series into its constituent parts. This involves identifying the predictable structure from the random variation. Once separated, each component can be examined and interpreted on its own terms.

We use the **additive model** here, which assumes the components add together: *observed = trend + seasonal + residual*. This is appropriate when the seasonal swings stay roughly the same size over time, regardless of the overall level, which is a reasonable assumption for London temperature.


<div class="alert alert-block alert-warning">
<b>Note:</b><br>
The decomposition process here assumes that there is only one seasonal pattern (annual) and that it does not change in size over time. If there were multiple seasonal patterns (e.g., daily, weekly, and annual) or if the seasonal pattern changed in size (e.g., larger seasonal swings in later years), we would need to use a more complex decomposition method that can handle these features. We will revisit this in later modules.
</div>

In [ ]:
# Decompose time series into components (annual seasonality: period=365 days)
stl = STL(daily_temp, period=365, robust=True)
decomposition = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 11), constrained_layout=True)

# Observed
daily_temp.plot(ax=axes[0], color='steelblue', linewidth=0.8)
axes[0].set_ylabel('Observed (°C)')
axes[0].set_title('Time Series Decomposition: London Temperature 2023-2026', fontweight='bold')

# Trend
decomposition.trend.plot(ax=axes[1], color='red', linewidth=1.5)
axes[1].set_ylabel('Trend (°C)')
axes[1].set_title('Trend: long-term average level')

# Seasonal
decomposition.seasonal.plot(ax=axes[2], color='green', linewidth=0.8)
axes[2].set_ylabel('Seasonal (°C)')
axes[2].set_title('Seasonal: annual repeating pattern')

# Residual (noise)
decomposition.resid.plot(ax=axes[3], color='grey', alpha=0.7, linewidth=0.6)
axes[3].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[3].set_ylabel('Residual (°C)')
axes[3].set_title('Residual (Noise): unpredictable day-to-day variation')
axes[3].set_xlabel('Date')

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.show()
plt.close()


<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- **Observed:** The original daily temperature data
- **Trend:** The long-term average level. This is fairly stable over the 3 years of data, with only a slight increase of 1.5°C
- **Seasonal:** The repeating annual cycle. This has a strong peak in summer (~+10°C above average), trough in winter (~−10°C below average)
- **Residual:** Random day-to-day variation not explained by trend or seasonality. This is approximately ±3–4°C, but with outliers up to ±12°C

**Key lesson:** Decomposition reveals predictable patterns (trend + seasonal) that models can exploit. The residual component is inherently unpredictable, so it can be used to set the lower bound on the forecast error.
</div>

Look at the Residual (Noise) panel at the bottom. In which season does the residual appear largest? That is, the most spread out from the zero line? What might explain this?

<details>
<summary>Solution</summary>

Winter months tend to show larger residuals, which can be seen as the noise is more spread out from zero. This may be because winter weather in London is more variable: passing Atlantic weather systems create larger day-to-day temperature swings compared to the more stable summer months. This means winter temperature forecasts will be more challenging to get to the same level of precision as summer forecasts.

</details>


**Conclusion:** Understanding which components are present in your data directly informs planning horizons:
- **Trend:** Long-term strategic decisions such as multi-year budgets, capacity expansion etc.
- **Seasonal:** Annual resource planning such as hiring seasonal staff, adjust hours, adjust budgets.
- **Residual:** Day-to-day operational flexibility such as understanding how to maintain buffer capacity for unexpected variations.


### Seasonal Patterns by Month

Decomposition gives us the annual cycle as a continuous curve. However, when the points are descrete enough, a bar chart is clearer than a line chart for showing the average value for each month.

We will use this approach to show the average temperature and precipitation for each calendar month, averaged across all three years of data. This is a common way to summarise seasonal patterns in a way that is easy to interpret for planning staffing levels or event scheduling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Monthly average temperature
monthly_avg_temp = daily_temp.groupby(daily_temp.index.month).mean()
monthly_avg_temp.index = months

monthly_avg_temp.plot.bar(
    ax=axes[0],
    color='steelblue',
    alpha=0.8,
    edgecolor='black',
    xlabel='Month',
    ylabel='Average Temperature (°C)',
    title='Seasonal Pattern: Temperature by Month',
    legend=False
)
axes[0].grid(True, alpha=0.3, axis='y')

# Monthly precipitation
daily_precip = weather_data['precipitation'].resample('D').sum()
monthly_avg_precip = daily_precip.groupby(daily_precip.index.month).mean()
monthly_avg_precip.index = months

monthly_avg_precip.plot.bar(
    ax=axes[1],
    color='coral',
    alpha=0.8,
    edgecolor='black',
    xlabel='Month',
    ylabel='Average Daily Precipitation (mm)',
    title='Seasonal Pattern: Precipitation by Month',
    legend=False
)
axes[1].grid(True, alpha=0.3, axis='y')

plt.show()
plt.close()

print("Temperature seasonal range:")
print(f"  Warmest month (avg): {monthly_avg_temp.max():.1f}°C (Month {monthly_avg_temp.idxmax()})")
print(f"  Coldest month (avg): {monthly_avg_temp.min():.1f}°C (Month {monthly_avg_temp.idxmin()})")


**Conclusion:** Clear seasonal patterns allow forward planning. Temperature peaks in July–August, making summer the natural window for outdoor events; precipitation is more evenly distributed across the year, with slightly higher totals in autumn and winter. This data directly supports annual scheduling decisions, with peak-season events in summer, scaled-down operations in winter.


### Cyclical and Noise Components

Decomposition gives us trend, seasonal, and residual elements, but there are two more concepts to understand before we start forecasting.

**Cyclical patterns** repeat without a fixed period. These can typically be seen in economic cycles, industry demand cycles, or large-scale public behaviour changes. They are harder to identify and forecast than seasonal patterns.

For example, a charity experiences recurring donation surges during major natural disasters or humanitarian crises. These may happen multiple times over several years but do not follow a fixed annual calendar. 

**Noise** is random, unpredictable variation. Looking at our residuals:
- Day-to-day temperature variation of ±3-4°C beyond what trend and seasonality explain
- It can be caused by passing weather systems, measurement variation, and genuinely random factors

**Conclusion:** Every forecast has irreducible uncertainty from noise. To manage this uncertainty, we need to build operational buffers and not plan to operate at exactly forecast capacity, as actual results will always deviate.

### Time Series Assumptions

Before forecasting, ensure your data meets these assumptions:

| Assumption | Why It Matters | What To Do If Violated |
|:-----------|:--------------|:------------------|
| **Regular spacing** | Most models assume equal time intervals | Aggregate irregular timestamps to regular intervals |
| **Past predicts future** | Historical patterns must continue forward | Monitor forecast accuracy and retrain models when patterns shift |

As we saw in Section 1, we can predict time series data using regression models using external predictors. These can be more robust to changes in the underlying patterns, as they rely on external signals rather than just past values. However, they also require more data and careful feature selection to perform well. We will explore this further in later modules. 

---

## Part 2: Forecasting Models

Now that we can describe time series patterns, the next step is to forecast future values. Decomposition told us *what* the data looks like, with a stable trend, strong seasonality, and a moderate residual. This section explores three approaches to forecasting, each suited to different characteristics in the data.

We begin with [**exponential smoothing**](https://otexts.com/fpp3/expsmooth.html), which is one of the most widely used forecasting methods in business. Its key advantage is that it requires no external predictor variables, unlike the regression models in Section 1, which needed forcasted weather measures, such as temperature, precipitation, and wind speed to make a prediction. Exponential smoothing learns entirely from the historical pattern of the variable itself.


### Exponential Smoothing and the Holt-Winters Method

**Exponential smoothing** gives recent observations more influence on the forecast than older ones, with each older observation receiving proportionally less weight. The diagram below shows this idea: this month's data contributes 50% of the weighted average, last month's contributes 25%, and so on with older data still contributing but with diminishing effect.


```mermaid
flowchart TD
    A["3+ Months Ago: decreasing..."] --> E[Weighted Average Forecast]
    B["2 Months Ago: 12.5% weight"] --> E
    C["1 Month Ago: 25% weight"] --> E
    D["This Month: 50% weight"] --> E
```


**Try it yourself:** Use the slider below to adjust α (the smoothing parameter) and observe its effect on the temperature series. 

- A high α (close to 1) makes the smoothed line closely follow the raw data. It is reactive, but noisy. 
- A low α (close to 0) produces a much smoother line, but is slower to respond to genuine changes. 

This is the core trade-off that Holt-Winters manages automatically when it fits the model.

In [ ]:
def _ses(series, alpha):
    """Simple (single) exponential smoothing."""
    s = [series.iloc[0]]
    for val in series.iloc[1:]:
        s.append(alpha * val + (1 - alpha) * s[-1])
    return pd.Series(s, index=series.index)

def _plot_ses(alpha=0.3):
    sample = daily_temp.loc['2024']  # one year: clear seasonal arc without too much noise
    smoothed = _ses(sample, alpha)

    fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
    ax.plot(sample.index, sample.values,
            label='Observed', color='steelblue', alpha=0.5, linewidth=0.9)
    ax.plot(smoothed.index, smoothed.values,
            label=f'Smoothed (α={alpha:.2f})', color='crimson', linewidth=2)
    ax.set_title(
        f'Simple Exponential Smoothing  |  α={alpha:.2f}  '
        f'(higher α → more reactive  •  lower α → smoother)'
    )
    ax.set_ylabel('Temperature (°C)')
    ax.set_xlabel('Date')
    ax.legend()
    plt.show()

alpha_slider = FloatSlider(
    min=0.01,
    max=0.99,
    step=0.01,
    value=0.3,
    description='α',
    continuous_update=False,
    layout=Layout(width='700px'),
    style={'description_width': '40px'}
)

ui = VBox(
    [alpha_slider],
    layout=Layout(width='100%', align_items='center')
)

out = interactive_output(_plot_ses, {'alpha': alpha_slider})
display(ui, out)

<div class="alert alert-block alert-info">
<b>What do you see?</b><br>

When we change the value of α, we see the smoothed line change in how closely it follows the raw data. 

- What are the advantages of a high α (close to 1)? How fast does it react to changes in the data? What are the disadvantages?
- What are the advantages of a low α (close to 0)? How smooth is the line? What are the disadvantages?

</div>

This weighting scheme means the model adapts quickly when patterns shift recent data matters most, while older data still contributes but with diminishing influence. If last summer was unusually warm, the model will adjust its summer forecast accordingly, but it will not completely ignore the summers before that.

There is a **lag** in the forecast when patterns shift, such as after a peak in temperature. This means when the values are rising, the forecast will tend to underestimate the true value, and when values are falling, it will tend to overestimate. The model is always trying to catch up with the latest data, which creates this lag effect. 

Smoothing the noise in this way also means reducing the size of the peaks and troughs in the data. This is a trade-off: we get a clearer view of the underlying trend and seasonality, but we lose some of the detail in the data. If the absolute values of the peaks and troughs are important for your forecasting goals, you may need to consider a different approach that retains more of the raw data's variability.

The **Holt-Winters method** extends basic exponential smoothing to handle all three predictable components simultaneously: the overall level, the direction of change (trend), and the repeating annual cycle (seasonality). It is one of the most practical forecasting tools available: straightforward to implement, fast to run, and effective across a wide range of business scenarios.


### Preparing Monthly Data

For forecasting monthly seasonal patterns, we first aggregate daily temperatures to monthly averages using `.resample('ME').mean()`. This gives us only one data point per month, reducing the noise for the model to learn from, and a natural match for the annual seasonal cycle. Smaller time intervals, such as weekly or daily would require us to think about how to handle varying month lengths and leap years.

The **training set** covers the first 30 months; the final 6 months are held back as a **test set**, representing data the model will never see during training.


In [ ]:
# Prepare monthly temperature data for forecasting
monthly_temp = weather_data['temperature_2m'].resample('ME').mean()  # Aggregate daily values to one average per month

# Split: train on first ~30 months, test on last 6 months
n_test = 6
train_data = monthly_temp.iloc[:-n_test]
test_data  = monthly_temp.iloc[-n_test:]

print(f"Training:  {len(train_data)} months ({train_data.index[0].strftime('%Y-%m')} to {train_data.index[-1].strftime('%Y-%m')})")
print(f"Testing:   {len(test_data)} months  ({test_data.index[0].strftime('%Y-%m')} to {test_data.index[-1].strftime('%Y-%m')})")

**Reinforcement Resource**

[Forecasting: Principles and Practice, Chapter 8 Exponential Smoothing (Hyndman and Athanasopoulos, 2021)](https://otexts.com/fpp3/expsmooth.html). This is an open-access textbook chapter explaining exponential smoothing concepts and the Holt-Winters method in detail. It uses examples coded in R, but the conceptual explanations transfer directly to this notebook.

*Read time: 15 minutes*


The next cell builds the Holt-Winters model. It has three key parameters:
- `seasonal_periods=12`: This tells the model to look for a pattern that repeats every 12 months, which matches the annual cycle we identified in the decomposition.
- `trend='add'`: This specifies that the trend component should be additive, meaning it adds a constant amount to the forecast each month. This is appropriate when the trend is linear and does not grow exponentially.
- `seasonal='add'`: This specifies that the seasonal component should be additive, meaning it adds a constant amount to the forecast each month. This is appropriate when the seasonal pattern is consistent over time.

In [ ]:
# Build Holt-Winters model (additive trend + additive seasonality)
model_hw = ExponentialSmoothing(
    train_data,
    seasonal_periods=12,   # 12 months = annual cycle
    trend='add',           # Additive trend component
    seasonal='add'         # Additive seasonality component
).fit()

# Forecast the 6 test months
forecast_hw = model_hw.forecast(steps=n_test)

test_data_df = test_data.to_frame()  # Create a DataFrame for test data to align with forecast results
test_data_df = test_data_df.rename(columns={'temperature_2m': 'Actual (°C)'})
test_data_df['Forecast (°C)'] = forecast_hw
test_data_df['Error (°C)'] = test_data_df['Actual (°C)'] - test_data_df['Forecast (°C)']

test_data_df.style.format({"Actual (°C)": "{:.1f}",
                           "Forecast (°C)": "{:.1f}",
                           "Error (°C)": "{:+.1f}",
                          }).format_index(lambda x: x.strftime("%Y-%m"), axis=0).set_caption("Holt-Winters Forecast vs Actual Temperatures")

**Conclusion:** The forecast was performed over a peried where the actual values were decreasing. We can see that the forecast captures the overall downward trend, with each predicted value being lower than the previous one, matching the actual data's pattern. However, the forecast has a negative error in every month, meaning it consistently overestimates the true values. This is likely due to the lag effect we discussed earlier: the model is always trying to catch up with the latest data, which creates a lag in the forecast when patterns shift. When values are falling, it will tend to overestimate, which may explain the consistent negative error in this case.

Let's visualise this in the context of the historical data. The next cell plots the monthly data with the forecast overlaid. 

In [ ]:
# Visualise: historical, forecast, and actual test data
fig, ax = plt.subplots(figsize=(14, 5), constrained_layout=True)

# Historical data
monthly_temp.plot(ax=ax, label='Historical', color='steelblue', alpha=0.7)

# Forecast
pd.Series(forecast_hw.values, index=test_data.index, name='Forecast').plot(
    ax=ax, label='Forecast (Holt-Winters)', color='red', 
    linestyle='--', marker='o', markersize=6)

# Actual test
test_data.plot(ax=ax, label='Actual (test period)', color='green', marker='s', markersize=6)

ax.set_xlabel('Date')
ax.set_ylabel('Average Monthly Temperature (°C)')
ax.set_title('Holt-Winters Exponential Smoothing Forecast: Monthly Temperature')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()
plt.close()


In [ ]:
# Evaluate forecast accuracy
mae  = mean_absolute_error(test_data, forecast_hw)
rmse = root_mean_squared_error(test_data, forecast_hw)
range = test_data.max() - test_data.min()

print(f"=== Forecast Accuracy ===")
print(f"  MAE:  {mae:.2f}°C   (average absolute error)")
print(f"  RMSE: {rmse:.2f}°C   (typical forecast error)")
print(f"\n  Interpretation: Monthly temperature forecasts are typically within ±{mae:.1f}°C of actual. This is {mae / range:.0%} of the observed temperature range during the test period.")

<div class="alert alert-block alert-info">
<b>What have we done?</b><br>

- We aggregated to monthly data for clearer seasonal modelling
- We trained Holt-Winters on 30 months, then forecast 6 months ahead
- The model captures the seasonal pattern (warmer summer forecasts, cooler winter forecasts)
- MAE ≈ 2–3°C means forecasts are reasonably accurate for monthly temperature

**Key conclusion:** Exponential smoothing requires no external predictor variables as it learns purely from historical temporal patterns. This is advantageous when future predictors are unavailable or uncertain. However, it also means the model is more reactive to recent changes, which can create a lag effect when patterns shift, leading to consistent overestimation or underestimation depending on the direction of change.
</div>

### Comparing Approaches: ARIMA

Holt-Winters works well when seasonality is clear, as it is for temperature. However, many workplace time series are driven more by **autocorrelation** than by obvious seasonal cycles. Autocorrelation means the current value is related to recent past values, a correlation between a time series and a delayed (lagged) version of itself. Understanding the strenght and pattern of autocorrelation in your data is crucial for choosing the right forecasting method. 

For example, every weekday, commuter traffic peaks in the morning and evening. Knowing the traffic volume at 8am on a Monday can help predict the traffic volume at 8am on a Tuesday. The traffic volume is highly autocorrelated with a lag of 1 day. Equally, knowing the traffic volume at 8am on a Monday can also help predict the traffic volume at 5pm in the opposite direction, as the same commuters return home. This is an example of a negative autocorrelation with a lag of 9 hours.

The next interactive visualisation helps you to explore autocorrelation directly using monthly temperature data:

- Use the **Lag** slider to shift the series forward or backward in time
- The **left plot** overlays the original and shifted series so you can compare their alignment across months
- The **right plot** shows a scatter of original values (x-axis) against shifted values (y-axis), with the correlation coefficient displayed
- Both plots update together, so you can see how changing lag affects alignment and correlation in real time

Before you start, try to predict how the plots will change as you adjust the lag. 

- When will the correlation be strongest? 
- When will it be weakest?
- When will it be negative?

In [ ]:
def _plot_autocorrelation_shift(lag=1):
    series = monthly_temp.rename('original')
    shifted = series.shift(lag).rename(f'shifted_{lag:+d}')
    aligned = pd.concat([series, shifted], axis=1).dropna()
    correlation = aligned.iloc[:, 0].corr(aligned.iloc[:, 1])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

    axes[0].plot(series.index, series.values, label='Original monthly temperature', color='steelblue', linewidth=2)
    axes[0].plot(shifted.index,
                 shifted.values,
                 label=f'Shifted by {lag:+d} month(s)',
                 color='darkorange',
                 linewidth=2,
                 linestyle='--')
    
    axes[0].set_title('Autocorrelation View: Series Overlaid Across Time')
    axes[0].set_xlabel('Date')
    axes[0].set_ylabel('Average Monthly Temperature (°C)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(aligned.iloc[:, 0],
                    aligned.iloc[:, 1],
                    color='teal',
                    edgecolor='white',
                    s=80,
                    alpha=0.85)

    if len(aligned) >= 2:
        slope, intercept = np.polyfit(aligned.iloc[:, 0], aligned.iloc[:, 1], 1)
        x_line = np.linspace(aligned.iloc[:, 0].min(), aligned.iloc[:, 0].max(), 100)
        axes[1].plot(x_line, slope * x_line + intercept, color='crimson', linewidth=2)

    axes[1].set_title(f'Scatter of Original vs Shifted Values | r = {correlation:.3f}')
    axes[1].set_xlabel('Original monthly temperature (°C)')
    axes[1].set_ylabel(f'Shifted monthly temperature (lag {lag:+d}) (°C)')
    axes[1].grid(True, alpha=0.3)
    axes[1].text(
        0.03,
        0.97,
        f'Correlation: {correlation:.3f}\nPaired months: {len(aligned)}',
        transform=axes[1].transAxes,
        va='top',
        bbox={'boxstyle': 'round', 'facecolor': 'white', 'alpha': 0.85, 'edgecolor': 'lightgrey'}
    )

    plt.show()
    plt.close(fig)

lag_slider = IntSlider(
    min=-12,
    max=12,
    step=1,
    value=1,
    description='Lag',
    continuous_update=False,
    layout=Layout(width='700px'),
    style={'description_width': '50px'}
)

lag_ui = VBox([lag_slider], layout=Layout(width='100%', align_items='center'))

lag_out = interactive_output(_plot_autocorrelation_shift, {'lag': lag_slider})
display(lag_ui, lag_out)

<details>
<summary>Visualisation Key relationships</summary>

| Lag (months) | Correlation | Explanation |
|:-------------:|:------------:|:------------|
|-12 | 0.94 | The series is compared to itself shifted by one year, so the seasonal pattern aligns well |
|-6 | -0.94 | The series is compared to itself shifted by half a year, so the peaks align with troughs, creating a negative correlation |
|-3 | -0.04 | The series is compared to itself shifted by three months, half way between the peaks and troughs. The correlation is at its lowest |
| 0 | 1.00 | Perfect correlation, as the series is compared to itself |
| 3 | -0.04 | The series is compared to itself shifted by three months, half way between the peaks and troughs. The correlation is at its lowest |
| 6 | -0.94 | The series is compared to itself shifted by half a year, so the peaks align with troughs, creating a negative correlation |
| 12 | 0.94 | The series is compared to itself shifted by one year, so the seasonal pattern aligns well |

</details>

[**ARIMA** (AutoRegressive Integrated Moving Average)](https://otexts.com/fpp3/arima.html) makes use of this feature. Holt-Winters estimated trend and seasonality directly, whereas ARIMA models make use of recent values to identify temporal dependencies. ARIMA models have three components that can be combined in different ways depending on the data's characteristics:

| Component | Key parameter | Meaning | When It Helps |
|:----------|:-------------:|:--------|:--------------|
| **AR (AutoRegressive)** | p | Current value depends on recent past values | Sales this week are influenced by last week's sales |
| **I (Integrated)** | d | Differencing to account for trends | Revenue growing or dropping steadily |
| **MA (Moving Average)** | q | Uses past forecast errors in predictions | Corrects for systematic over/under-prediction (bias) |

ARIMA models are described using the three parameters (p, d, q), often written as ARIMA(p,d,q). These parameters define the structure of the model and are always non-negative integers. 

In the cell below, we use ARIMA(1,2,0): one autoregressive term, one round of differencing, and one moving average term. This is a common starting point for exploratory modelling.

In [ ]:
# Fit a simple ARIMA(1,2,0) model to monthly temperature
# (1,2,0) = 1 autoregressive term, 2 differences, 0 moving average terms
model_arima = ARIMA(train_data, order=(1, 2, 0))
model_arima_fit = model_arima.fit()

# Forecast 6 months ahead
forecast_arima = model_arima_fit.forecast(steps=n_test)

# Compare ARIMA and Holt-Winters
mae_arima  = mean_absolute_error(test_data, forecast_arima)
rmse_arima = root_mean_squared_error(test_data, forecast_arima)

print("=== Model Comparison: Holt-Winters vs ARIMA ===")
print(f"\nHolt-Winters: MAE = {mae:.2f}°C,  RMSE = {rmse:.2f}°C")
print(f"ARIMA(1,2,0): MAE = {mae_arima:.2f}°C,  RMSE = {rmse_arima:.2f}°C")

# Visualise comparison
fig, ax = plt.subplots(figsize=(14, 5), constrained_layout=True)

# Historical data
monthly_temp.plot(ax=ax, label='Historical', color='steelblue', alpha=0.7)

# Actual test data
test_data.plot(ax=ax, label='Actual (test)', color='black', marker='s', markersize=7, zorder=5)

# Holt-Winters forecast
pd.Series(forecast_hw.values, index=test_data.index).plot(ax=ax, label=f'Holt-Winters (MAE={mae:.1f}°C)', 
                                                          color='red', linestyle='--', marker='o', 
                                                          markersize=6)

# ARIMA forecast
pd.Series(forecast_arima.values, index=test_data.index).plot(ax=ax, label=f'ARIMA (MAE={mae_arima:.1f}°C)', 
                                                             color='purple', linestyle=':', marker='^', 
                                                             markersize=6)

ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Forecast Comparison: Holt-Winters vs ARIMA')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()
plt.close()


<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We fitted a simple ARIMA(1,2,0) model and compared it to Holt-Winters
- We start our prediction in August 2025, when we would expect the seasonal peak to have passed and the values to be falling
- The Holt-Winters model appears to follow the seasonal pattern on a delay, while the ARIMA model continues to predict growth beyond the seasonal peak. This is likely because:
    - the ARIMA model is picking up on the recent upward trend in the data, which continues into the forecast period
    - the Holt-Winters model has been told to expect a seasonal peak every 12 months, so it is predicting a decline in line with the changes seen in 2024.  
- For data with one strongly seasonal cycle like yearly temperature, Holt-Winters typically performs better because it explicitly models the seasonal component. However, ARIMA can be extended to seasonal ARIMA (SARIMA) for seasonal data. This comes with the cost of a model with more parameters to tune, which can be challenging with limited data. 
- We will revisit ARIMA and SARIMA in later modules, as they are useful tools for forecasting complex time series data.

**Key lesson:** No single forecasting model wins in all situations. In practice, compare multiple approaches using your test data before choosing one for production use.
</div>

Let's summarise the differences in the four models we have explored:

```mermaid
flowchart TD
    A[Time Series Forecasting] --> ExternalPredictors{External predictors available?}
    ExternalPredictors -- Yes --> Regression[Regression-like approaches]
    ExternalPredictors -- No --> Autocorrelation{Strong autocorrelation?}
    Autocorrelation -- Yes --> SeasonalPatternCorr{Clear seasonal pattern?}
    Autocorrelation -- No --> SeasonalPatternNoCorr{Clear seasonal pattern?}
    SeasonalPatternCorr -- Yes --> SARIMA[SARIMA]
    SeasonalPatternCorr -- No --> ARIMA[ARIMA]
    SeasonalPatternNoCorr -- Yes --> HoltWinters[Holt-Winters]
    SeasonalPatternNoCorr -- No --> SimpleSmoothing[Simple Exponential Smoothing]
```


The choice of model depends on your data's characteristics: the strength of seasonal patterns, whether holidays impact the series, the complexity of temporal dependencies, and whether external predictor variables are available. This decision tree gives you a starting point for selecting an approach. However, in practice, you would typically trial two or three candidates and compare their accuracy on a held-out test set.


## Advanced Models

We have only scratched the surface of time series forecasting methods. Here are two more advanced tools that are worth being aware of, even if they are beyond the scope of this module. If your data has overlapping seasonal patterns at multiple levels, such as, hourly customer contact patterns alongside weekly demand cycles and annual holiday effects, these models may deliver improved forecast accuracy over the approaches explored here.

### Prophet (Meta / Facebook)

**Prophet** (Taylor and Letham, 2018) was designed for business time series with multiple seasonality levels, holiday effects, and missing data. It handles:
- **Multiple seasonality levels:** Daily, weekly, and yearly patterns simultaneously
- **Custom holidays:** Easter, Eid, Black Friday and other events that change date each year
- **Missing data:** Handles gaps in data gracefully

**Use cases:** Retail sales forecasting incorporating daily patterns, weekly cycles, annual seasonality, and UK public holidays simultaneously.

**Link:** [https://facebook.github.io/prophet/](https://facebook.github.io/prophet/)

### Greykite (LinkedIn)

**Greykite** (Hosseini et al., 2021) builds forecasts faster than Prophet while retaining the same capacity for complex seasonality and holiday effects. It also includes anomaly detection capabilities.

**Use cases:** High-frequency operational data (hourly call volumes, daily transactions) where forecast speed matters for real-time decision-making.

**Link:** [https://linkedin.github.io/greykite/](https://linkedin.github.io/greykite/)

---

## Part 3: Modified Task: Forecast Monthly Precipitation

You have now seen Holt-Winters applied to temperature, which is a variable with a strong, predictable seasonal pattern. But not all data is this well-behaved. In this task, you will apply the same method to precipitation and explore what happens when the seasonal signal is weaker.

This is one important difference we have to consider for precipitation: We are interested in the total monthly rainfall, that is how much rain fell across the whole month, rather than the average hourly rainfall rate. We will therefore need to aggregate the hourly precipitation data differently than we did for temperature. Instead of taking the mean, using `.resample('ME').mean()`, we will take the sum, using `.resample('ME').sum()`. This choice of aggregation method (sum vs mean) affects how we interpret the results: the forecast will predict total millimetres of rain per month, not an average rate.


<div class="alert alert-block alert-success">
<b>Your turn:</b><br>
Build a Holt-Winters model to forecast monthly precipitation for the 6-month test period. Is precipitation easier or harder to forecast than temperature? 

The steps are the same as the ones we went through for temperature in the "Part 2: Forecasting Models -> Preparing Monthly Data" section.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Steps:
1. Create `monthly_precip` from `weather_data['precipitation'].resample('ME').sum()` (sum, not mean — we want total monthly rain)
2. Split into train (all except last 6 months) and test (last 6 months)
3. Fit `ExponentialSmoothing` with `seasonal_periods=12`, `trend='add'`, `seasonal='add'`
4. Forecast 6 months ahead
5. Calculate MAE and RMSE, then visualise

</details>
</div>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
# Prepare monthly precipitation data for forecasting
monthly_precip = weather_data['precipitation'].resample('ME').sum()  # Aggregate daily values to one total per month

# Split: train on first ~30 months, test on last 6 months
n_test = 6
train_data = monthly_precip.iloc[:-n_test]
test_data  = monthly_precip.iloc[-n_test:]

# Build Holt-Winters model (additive trend + additive seasonality)
model_hw = ExponentialSmoothing(
    train_data,
    seasonal_periods=12,   # 12 months = annual cycle
    trend='add',           # Additive trend component
    seasonal='add'         # Additive seasonality component
).fit()

# Forecast the 6 test months
forecast_hw = model_hw.forecast(steps=n_test)

test_data_df = test_data.to_frame()  # Create a DataFrame for test data to align with forecast results
test_data_df = test_data_df.rename(columns={'precipitation': 'Actual (mm)'})
test_data_df['Forecast (mm)'] = forecast_hw
test_data_df['Error (mm)'] = test_data_df['Actual (mm)'] - test_data_df['Forecast (mm)']

test_data_df.style.format({"Actual (mm)": "{:.1f}",
                           "Forecast (mm)": "{:.1f}",
                           "Error (mm)": "{:+.1f}",
                          }).format_index(lambda x: x.strftime("%Y-%m"), axis=0).set_caption("Holt-Winters Forecast vs Actual Precipitation")


# Visualise: historical, forecast, and actual test data
fig, ax = plt.subplots(figsize=(14, 5), constrained_layout=True)

# Historical data
monthly_precip.plot(ax=ax, label='Historical', color='steelblue', alpha=0.7)

# Forecast
pd.Series(forecast_hw.values, index=test_data.index, name='Forecast').plot(
    ax=ax, label='Forecast (Holt-Winters)', color='red', 
    linestyle='--', marker='o', markersize=6)

# Actual test
test_data.plot(ax=ax, label='Actual (test period)', color='green', marker='s', markersize=6)

ax.set_xlabel('Date')
ax.set_ylabel('Monthly Precipitation (mm)')
ax.set_title('Holt-Winters Exponential Smoothing Forecast: Monthly Precipitation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()
plt.close()

# Evaluate forecast accuracy
mae  = mean_absolute_error(test_data, forecast_hw)
rmse = root_mean_squared_error(test_data, forecast_hw)
range = test_data.max() - test_data.min()

print(f"=== Forecast Accuracy ===")
print(f"  MAE:  {mae:.2f} mm   (average absolute error)")
print(f"  RMSE: {rmse:.2f} mm   (typical forecast error)")
```
</details>

In [ ]:
# Your code here for the forecast monthly precipitation




**Expected outcome:** Precipitation MAE will likely be a greater proportion of the range than we found with temperature. This is to be expected as precipitation is more variable and depends heavily on passing weather systems that are inherently chaotic, whereas temperature follows a strong, consistent seasonal cycle.

**Success check:** You can explain why temperature is more forecastable than precipitation? 

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

    Temperature follows a strong, consistent seasonal cycle; precipitation is more variable and depends heavily on passing weather systems that are inherently chaotic.
</details>


---

## Part 4: Independent Task: Volunteer Forecasting

This final task asks you to apply to a new scenario everything from this section: time series components, forecasting methods, and model selection. As with Section 1's independent task, the focus is on analytical planning: designing the right approach before any code is written. 

Start with the time series components. What patterns would you expect to see in volunteer sign-up data? One you have some ideas, then work through each section of the template. Aim for approximately 200 words across the template fields.


### Your Challenge

You work for a charity managing volunteer programmes. Volunteer sign-up numbers vary throughout the year. Design a time series forecasting approach to predict monthly volunteer numbers three months ahead.

**Consider:**

1. **Time Series Components:**
   - What trend might exist? (Growing as charity expands? Stable? Declining?)
   - What seasonal patterns exist? (University term times? December giving season? Summer lull during holidays?)
   - What creates unpredictable variation? (Major news events triggering surge volunteering?)

2. **Forecasting Approach:**
   - Which method suits this data? (Holt-Winters for clear seasonality? ARIMA for autocorrelation?)
   - What forecast horizon is useful operationally? (3 months enables coordinator planning)
   - What accuracy is "good enough"? (±20% may be sufficient if programmes are flexible)

3. **Operational Use:**
   - How do forecasts inform coordinator decisions?
   - What decisions change based on predicted volunteer numbers?
   - Which forecast error is costlier: over-predicting or under-predicting?

4. **Data Requirements:**
   - How many years of historical data are needed?
   - What additional variables might improve forecasts (campaign launch dates, university term dates)?
   - How to handle major one-off events (national volunteering campaigns)?


<div class="alert alert-block alert-success">
<b>Your turn:</b><br>
Complete the time series forecasting plan below.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

**Components hints:**
- Trend: Possibly growing if charity is expanding its reach
- Seasonal: September spike (university students return), December spike (festive giving spirit), summer dip (holidays)
- Noise: Major disasters or media coverage triggers unforeseeable surges

**Approach hints:**
- Holt-Winters handles trend + seasonality well; good starting point
- 3-month horizon: sufficient for coordinator scheduling and training resource planning
- ±20% accuracy likely sufficient: volunteer programmes can flex capacity

**Operational hints:**
- Under-forecast (100 predicted, 130 show up): Insufficient materials, overwhelmed coordinators, poor volunteer experience
- Over-forecast (100 predicted, 70 show up): Wasted training resources, underutilised prepared activities
- Both errors have real costs; identify which matters more to your charity's mission
</details>
</div>


In [ ]:
# Your forecasting plan (complete for portfolio evidence)
volunteer_forecast_plan = {
    'time_series_components': {
        'trend': '',
        'seasonal_patterns': [],
        'unpredictable_factors': []
    },
    'forecasting_approach': {
        'method': '',
        'justification': '',
        'forecast_horizon': '',
        'accuracy_target': ''
    },
    'operational_use': {
        'decisions_informed': [],
        'cost_over_forecast': '',
        'cost_under_forecast': ''
    },
    'data_requirements': {
        'historical_data_needed': '',
        'additional_variables': [],
        'special_events_handling': ''
    }
}

# Display your plan
print("=== VOLUNTEER PROGRAMME FORECASTING PLAN ===")
for section, details in volunteer_forecast_plan.items():
    print(f"\n{section.upper().replace('_', ' ')}:")
    if isinstance(details, dict):
        for k, v in details.items():
            print(f"  {k}: {v}")
    else:
        print(f"  {details}")

**Portfolio note:** This time series forecasting plan demonstrates your understanding of temporal data analysis and your ability to apply data-gathering methods to real workplace problems (MLO 3). Include it in your Module Challenge Part A portfolio. Your plan should clearly identify the components present in your data, justify your choice of forecasting method, and quantify the operational value of improved forecast accuracy.


---

## Reflection

Reflect on the following two questions and record your thoughts in the cell below. Ground your answers in examples from your workplace, using **approximately 300 words** in total.

Your answers here form part of your Module Challenge Part A portfolio and are assessed against **MLO3:** *Select and apply appropriate data-gathering methods for a given data analysis scenario*.

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your responses. You can develop your answers further before your final submission.

<div class="alert alert-block alert-info">

1. Exponential smoothing requires no external predictors; it learns from historical patterns alone. When would this be advantageous over the regression approach from Section 1, which needed external predictor data? Give an example from your sector.

2. Imagine your volunteer forecast is 30% too low (100 predicted, 130 show up) versus 30% too high (100 predicted, 70 show up). Which error is more costly in your charity context? Does this asymmetry affect how you would set forecast thresholds or build operational buffers?

</div>

*Write your responses here.*

1. 

2. 

References (if applicable):

---

## Summary

**You have completed Week 7 Section 2.** You can now:

- Identified the four time series components (trend, seasonal, cyclical, noise) and used STL decomposition to separate London's daily temperature series into trend, seasonal, and residual parts
- Built a Holt-Winters exponential smoothing model on monthly temperature data and compared its forecast accuracy against an ARIMA(1,2,0) model using MAE and RMSE
- Applied Holt-Winters to monthly precipitation totals and explained why precipitation is harder to forecast than temperature due to its weaker seasonal signal
- Designed a time series forecasting plan for a charity's volunteer sign-up numbers, applying the same component-then-method reasoning independently

**Key takeaways:**

- Exponential smoothing methods such as Holt-Winters require no external predictor variables: they forecast purely from a variable's own historical pattern, unlike the regression approach in Section 1.
- Holt-Winters lags behind sudden changes in direction, so it tends to overestimate values that are falling and underestimate values that are rising.

**What comes next:**

- This concludes the taught weekly content for DT401 ahead of finalising your Module Challenge Part A portfolio. The Workshops (for example, Workshop 1's SQL practice) remain available for further hands-on practice with these techniques.

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 7 Section 2 Analysing Data Through Time"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.7 component on Aptem

---

**Reference list for this section:**
- Hosseini, R. et al. (2021) *A flexible forecasting model for production systems*. arXiv. Available at: https://doi.org/10.48550/arXiv.2105.01098
- Hyndman, R.J. and Athanasopoulos, G. (2021) *Forecasting: Principles and Practice*. 3rd edn. Melbourne: OTexts. Available at: https://otexts.com/fpp3/
- Open-Meteo (2026) *Historical Weather API*. Available at: https://open-meteo.com [Accessed 27 February 2026]
- Taylor, S.J. and Letham, B. (2018) 'Forecasting at Scale', *The American Statistician*, 72(1), pp. 37–45. https://doi.org/10.1080/00031305.2017.1380080